- Author: Yousef Al Zeer

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns',100)
import missingno
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
# Set pandas as the default output for sklearn
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
# Load data directly from url
df = pd.read_csv('https://docs.google.com/spreadsheets/d/1jfU2oFSfhX1ywUbqETExDJuztO95r3h6pbWAm7xpwNY/gviz/tq?tqx=out:csv&sheet=users')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sex             4177 non-null   object 
 1   length          4177 non-null   float64
 2   diameter        4177 non-null   float64
 3   height          4177 non-null   float64
 4   whole_weight    4177 non-null   float64
 5   shucked_weight  4177 non-null   float64
 6   viscera_weight  4177 non-null   float64
 7   shell_weight    4177 non-null   float64
 8   rings           4177 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 293.8+ KB


## Exploring Data

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.isna().sum()

,0
sex,0
length,0
diameter,0
height,0
whole_weight,0
shucked_weight,0
viscera_weight,0
shell_weight,0
rings,0


In [ ]:
df['sex'].value_counts()

,count
sex,
M,1528
I,1342
F,1307


In [ ]:
df.describe()

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
count,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000
mean,0.523992,0.407881,0.139516,0.828742,0.359367,0.180594,0.238831,9.933684
std,0.120093,0.099240,0.041827,0.490389,0.221963,0.109614,0.139203,3.224169
min,0.075000,0.055000,0.000000,0.002000,0.001000,0.000500,0.001500,1.000000
25%,0.450000,0.350000,0.115000,0.441500,0.186000,0.093500,0.130000,8.000000
50%,0.545000,0.425000,0.140000,0.799500,0.336000,0.171000,0.234000,9.000000
75%,0.615000,0.480000,0.165000,1.153000,0.502000,0.253000,0.329000,11.000000
max,0.815000,0.650000,1.130000,2.825500,1.488000,0.760000,1.005000,29.000000


In [ ]:
# upper_bound = Q3 + 1.5 * IQR
unusual_height = df[df['height'] > 0.24]
unusual_height

,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
1417,M,0.705,0.565,0.515,2.2100,1.1075,0.4865,0.5120,10
1428,F,0.815,0.650,0.250,2.2550,0.8905,0.4200,0.7975,14
1763,M,0.775,0.630,0.250,2.7795,1.3485,0.7600,0.5780,12
2051,F,0.455,0.355,1.130,0.5940,0.3320,0.1160,0.1335,8
2179,F,0.595,0.470,0.250,1.2830,0.4620,0.2475,0.4450,14


In [ ]:
zero_height_rows = df['height'] == 0
df[zero_height_rows]

,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
1257,I,0.430,0.34,0.0,0.428,0.2065,0.0860,0.1150,8
3996,I,0.315,0.23,0.0,0.134,0.0575,0.0285,0.3505,6


Considering unusual cases:

- We have 2 rows where the height is zero, which is completely illogical.

- We have several rows, one of which has a significant gap between the length and height. This isn't an outlier, but rather an anomalous value.

We have a total of 3 rows that need processing. Since they are only 3 out of 4177, the best strategy is to drop them. This is because we are also dealing with very small range data. Any imputation, even with a median, will result in basing.

In [ ]:
df = df[(df['height'] != 0) & (df['height'] != 1.130)]

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4174 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sex             4174 non-null   object 
 1   length          4174 non-null   float64
 2   diameter        4174 non-null   float64
 3   height          4174 non-null   float64
 4   whole_weight    4174 non-null   float64
 5   shucked_weight  4174 non-null   float64
 6   viscera_weight  4174 non-null   float64
 7   shell_weight    4174 non-null   float64
 8   rings           4174 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 326.1+ KB


In [ ]:
df.describe()

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
count,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000
mean,0.524081,0.407953,0.139346,0.829061,0.359483,0.180668,0.238859,9.935553
std,0.120079,0.099228,0.038811,0.490395,0.221980,0.109614,0.139219,3.224474
min,0.075000,0.055000,0.010000,0.002000,0.001000,0.000500,0.001500,1.000000
25%,0.450000,0.350000,0.115000,0.442125,0.186125,0.093500,0.130000,8.000000
50%,0.545000,0.425000,0.140000,0.800000,0.336000,0.171000,0.234000,9.000000
75%,0.615000,0.480000,0.165000,1.153750,0.502000,0.253000,0.328875,11.000000
max,0.815000,0.650000,0.515000,2.825500,1.488000,0.760000,1.005000,29.000000


In [ ]:
df.describe(include='O')

,sex
count,4174
unique,3
top,M
freq,1528


## Train-Test Split

In [ ]:
target = 'rings'
X = df.drop(columns=target)
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3130 entries, 2026 to 860
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sex             3130 non-null   object 
 1   length          3130 non-null   float64
 2   diameter        3130 non-null   float64
 3   height          3130 non-null   float64
 4   whole_weight    3130 non-null   float64
 5   shucked_weight  3130 non-null   float64
 6   viscera_weight  3130 non-null   float64
 7   shell_weight    3130 non-null   float64
dtypes: float64(7), object(1)
memory usage: 220.1+ KB


## Define Tuples

### Numeric

In [ ]:
num_cols = ['length' , 'diameter' , 'height' , 'whole_weight' , 'shucked_weight' , 'viscera_weight' , 'shell_weight']
scaler = StandardScaler()


num_tuple = ('Numeric' , scaler , num_cols)
num_tuple

('Numeric',
 StandardScaler(),
 ['length',
  'diameter',
  'height',
  'whole_weight',
  'shucked_weight',
  'viscera_weight',
  'shell_weight'])

### Categorical

In [ ]:
cat_cols = ['sex']
ohe = OneHotEncoder(sparse_output=False , handle_unknown='ignore')
ohe_tuple = ('Categorical' , ohe , cat_cols)
ohe_tuple

('Categorical',
 OneHotEncoder(handle_unknown='ignore', sparse_output=False),
 ['sex'])

## ColumnTransformer

In [ ]:
col_transformer = ColumnTransformer([num_tuple , ohe_tuple ], verbose_feature_names_out=False)

In [ ]:
col_transformer.fit(X_train)

ColumnTransformer(transformers=[('Numeric', StandardScaler(),
                                 ['length', 'diameter', 'height',
                                  'whole_weight', 'shucked_weight',
                                  'viscera_weight', 'shell_weight']),
                                ('Categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['sex'])],
                  verbose_feature_names_out=False)

In [ ]:
X_train_processed = col_transformer.transform(X_train)
X_test_processed = col_transformer.transform(X_test)
print("training set processed : \n")
display(X_train_processed.head())
print("\n")
print("testing set processed : \n")
display(X_test_processed.head())

training set processed : 



,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_F,sex_I,sex_M
2026,0.291168,0.012141,0.531982,-0.020086,-0.085603,0.027978,0.099678,1.0,0.0,0.0
1068,-1.311181,-1.315900,-1.030029,-1.245537,-1.099602,-1.414079,-1.269433,0.0,1.0,0.0
247,-1.395515,-1.366978,-1.420532,-1.293494,-1.288463,-1.276740,-1.222472,0.0,1.0,0.0
1451,-0.552173,-0.549722,-0.769694,-0.806783,-0.735372,-0.777743,-0.825104,0.0,1.0,0.0
1603,0.122500,0.216455,0.011312,-0.024167,0.145976,-0.077315,-0.070107,0.0,1.0,0.0




testing set processed : 



,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_F,sex_I,sex_M
501,0.797173,1.135868,2.224162,0.718655,0.076278,0.815387,1.125608,1.0,0.0,0.0
463,-2.576193,-2.490705,-2.201538,-1.585317,-1.526786,-1.546840,-1.583715,0.0,1.0,0.0
1425,1.767016,1.595575,1.703491,2.645093,2.643877,2.555011,2.353834,1.0,0.0,0.0
1462,-0.257004,-0.294330,-0.509359,-0.527204,-0.429598,-0.480175,-0.687832,1.0,0.0,0.0
3026,-0.257004,-0.345408,-0.639527,-0.661892,-0.539767,-0.713651,-0.644483,0.0,1.0,0.0


In [ ]:
X_train_processed.dtypes

,0
length,float64
diameter,float64
height,float64
whole_weight,float64
shucked_weight,float64
viscera_weight,float64
shell_weight,float64
sex_F,float64
sex_I,float64
sex_M,float64


In [ ]:
print('Original training data: \n')
print(X_train.head())
print('------------------------')
print('Preprocessed training data: \n')
print(X_train_processed.head())

Original training data: 

     sex  length  diameter  height  whole_weight  shucked_weight  \
2026   F    0.56     0.410   0.160        0.8215          0.3420   
1068   I    0.37     0.280   0.100        0.2210          0.1165   
247    I    0.36     0.275   0.085        0.1975          0.0745   
1451   I    0.46     0.355   0.110        0.4360          0.1975   
1603   I    0.54     0.430   0.140        0.8195          0.3935   

      viscera_weight  shell_weight  
2026          0.1840        0.2530  
1068          0.0265        0.0635  
247           0.0415        0.0700  
1451          0.0960        0.1250  
1603          0.1725        0.2295  
------------------------
Preprocessed training data: 

        length  diameter    height  whole_weight  shucked_weight  \
2026  0.291168  0.012141  0.531982     -0.020086       -0.085603   
1068 -1.311181 -1.315900 -1.030029     -1.245537       -1.099602   
247  -1.395515 -1.366978 -1.420532     -1.293494       -1.288463   
1451 -0.552173 -